# Text Classifier With Embeddings

This notebook teaches the bridge from normal text to neural-network inputs.

The goal is:

```text
review sentence -> positive or negative
```

## 1. Import Libraries

We use normal Python tools for text/data handling, then PyTorch for the neural network.

In [1]:
import csv
import re
from collections import Counter
from pathlib import Path

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset, random_split

## 2. Load The Dataset

Each row has:

- `text`: a short review
- `label`: `1` for positive, `0` for negative

In [2]:
DATA_PATH = Path("data/reviews_tiny.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("11_embeddings_text_classifier/data/reviews_tiny.csv")


with DATA_PATH.open(newline="") as file:
    rows = [
        {"text": row["text"], "label": int(row["label"])}
        for row in csv.DictReader(file)
    ]

rows[:5]

[{'text': 'this movie was fantastic and fun', 'label': 1},
 {'text': 'i loved the story and the characters', 'label': 1},
 {'text': 'the acting was great and the ending was beautiful', 'label': 1},
 {'text': 'what a wonderful and exciting film', 'label': 1},
 {'text': 'this was a smart and emotional story', 'label': 1}]

## 3. Tokenization

A neural network cannot directly understand a sentence string.

First we split text into tokens:

```text
"this movie was fantastic" -> ["this", "movie", "was", "fantastic"]
```

In [3]:
def tokenize(text):
    return re.findall(r"[a-z']+", text.lower())

example = rows[0]["text"]
example, tokenize(example)

('this movie was fantastic and fun',
 ['this', 'movie', 'was', 'fantastic', 'and', 'fun'])

## 4. Vocabulary

A vocabulary maps each token to an integer.

We reserve:

- `0` for padding
- `1` for unknown words

In [4]:
PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"

counts = Counter()
for row in rows:
    counts.update(tokenize(row["text"]))

vocab = {PAD_TOKEN: 0, UNK_TOKEN: 1}
for token, count in sorted(counts.items()):
    vocab[token] = len(vocab)

list(vocab.items())[:20]

[('<pad>', 0),
 ('<unk>', 1),
 ('a', 2),
 ('absolutely', 3),
 ('accurate', 4),
 ('acting', 5),
 ('after', 6),
 ('am', 7),
 ('and', 8),
 ('app', 9),
 ('are', 10),
 ('awful', 11),
 ('bad', 12),
 ('balanced', 13),
 ('beautiful', 14),
 ('blurry', 15),
 ('book', 16),
 ('boring', 17),
 ('bright', 18),
 ('brilliant', 19)]

## 5. Encode Text

Encoding turns tokens into token IDs.

```text
["this", "movie", "was"] -> [token_id, token_id, token_id]
```

In [5]:
def encode_text(text, vocab):
    return [vocab.get(token, vocab[UNK_TOKEN]) for token in tokenize(text)]

encode_text("this movie was fantastic", vocab)

[106, 78, 120, 48]

## 6. Dataset

A PyTorch dataset returns one example at a time.

Each example is:

```text
token_ids, label
```

In [6]:
class ReviewDataset(Dataset):
    def __init__(self, rows, vocab):
        self.examples = [
            (
                torch.tensor(encode_text(row["text"], vocab), dtype=torch.long),
                torch.tensor(row["label"], dtype=torch.float32),
            )
            for row in rows
        ]

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        return self.examples[index]

dataset = ReviewDataset(rows, vocab)
dataset[0]

(tensor([106,  78, 120,  48,   8,  57]), tensor(1.))

## 7. Padding

Different sentences have different lengths.

A batch needs a rectangle-shaped tensor, so shorter sentences get padded with `0`.

In [7]:
def collate_batch(batch):
    token_ids, labels = zip(*batch)
    lengths = torch.tensor([len(ids) for ids in token_ids], dtype=torch.long)
    max_length = lengths.max().item()

    padded = torch.zeros((len(token_ids), max_length), dtype=torch.long)
    for row_index, ids in enumerate(token_ids):
        padded[row_index, : len(ids)] = ids

    labels = torch.stack(labels).unsqueeze(1)
    return padded, lengths, labels

sample_token_ids, sample_lengths, sample_labels = collate_batch([dataset[0], dataset[1], dataset[2]])
sample_token_ids, sample_lengths, sample_labels

(tensor([[106,  78, 120,  48,   8,  57,   0,   0,   0],
         [ 66,  75, 104, 101,   8, 104,  25,   0,   0],
         [104,   5, 120,  59,   8, 104,  42, 120,  14]]),
 tensor([6, 7, 9]),
 tensor([[1.],
         [1.],
         [1.]]))

## 8. Embedding Model

`nn.Embedding` learns a vector for each token ID.

The model does this:

```text
token IDs -> embedding vectors -> average vector -> classifier -> sentiment logit
```

In [8]:
class EmbeddingSentimentClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim=32, hidden_size=32):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.classifier = nn.Sequential(
            nn.Linear(embedding_dim, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 1),
        )

    def forward(self, token_ids, lengths):
        embeddings = self.embedding(token_ids)
        mask = (token_ids != 0).unsqueeze(-1)
        summed = (embeddings * mask).sum(dim=1)
        averaged = summed / lengths.unsqueeze(1).clamp(min=1)
        return self.classifier(averaged)

model = EmbeddingSentimentClassifier(vocab_size=len(vocab))
model

EmbeddingSentimentClassifier(
  (embedding): Embedding(127, 32, padding_idx=0)
  (classifier): Sequential(
    (0): Linear(in_features=32, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=1, bias=True)
  )
)

## 9. Training

The training loop is the same pattern you have seen before:

```text
predict -> loss -> backward -> optimizer step
```

In [9]:
def accuracy_from_logits(logits, labels):
    predictions = (torch.sigmoid(logits) >= 0.5).float()
    return (predictions == labels).float().mean().item()

torch.manual_seed(42)
generator = torch.Generator().manual_seed(42)
train_size = int(len(dataset) * 0.8)
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size], generator=generator)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_batch)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, collate_fn=collate_batch)

model = EmbeddingSentimentClassifier(vocab_size=len(vocab))
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(20):
    model.train()
    total_loss = 0.0
    total_accuracy = 0.0
    total_batches = 0

    for token_ids, lengths, labels in train_loader:
        logits = model(token_ids, lengths)
        loss = loss_fn(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_accuracy += accuracy_from_logits(logits, labels)
        total_batches += 1

    if epoch % 5 == 0 or epoch == 19:
        print(
            f"epoch {epoch + 1:2d}/20 | "
            f"loss {total_loss / total_batches:.4f} | "
            f"accuracy {total_accuracy / total_batches:.3f}"
        )

epoch  1/20 | loss 0.6973 | accuracy 0.575
epoch  6/20 | loss 0.4482 | accuracy 0.850
epoch 11/20 | loss 0.0772 | accuracy 1.000
epoch 16/20 | loss 0.0100 | accuracy 1.000
epoch 20/20 | loss 0.0043 | accuracy 1.000


## 10. Predictions

Now we can send new sentences through the same tokenizer, vocabulary, embeddings, and classifier.

In [10]:
examples = [
    "this movie was fun and beautiful",
    "the product is broken and awful",
    "the lesson was clear and helpful",
    "the app feels slow and confusing",
]

encoded_examples = [
    (torch.tensor(encode_text(text, vocab), dtype=torch.long), torch.tensor(0.0))
    for text in examples
]
token_ids, lengths, _ = collate_batch(encoded_examples)

model.eval()
with torch.no_grad():
    probabilities = torch.sigmoid(model(token_ids, lengths)).squeeze(1)

for text, probability in zip(examples, probabilities):
    label = "positive" if probability.item() >= 0.5 else "negative"
    print(f"{text!r} -> {label} ({probability.item():.3f})")

'this movie was fun and beautiful' -> positive (1.000)
'the product is broken and awful' -> negative (0.208)
'the lesson was clear and helpful' -> positive (1.000)
'the app feels slow and confusing' -> negative (0.000)
